In [42]:
import pandas as pd
from sqlalchemy import create_engine, text
import pymysql
from urllib.parse import quote_plus

# Connection details
username = "root"
password = "Admin@123"
host = "localhost"
port = 3306
database = "ai_jobs"

# Password ko encode karo (@ jaise characters handle karne ke liye)
encoded_password = quote_plus(password)

# Connection string banao - encoded_password use karo
engine = create_engine(f"mysql+pymysql://{username}:{encoded_password}@{host}:{port}/{database}")

# Test connection
try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT 1"))
        print("MySQL Connection successful!")
        print("Result:", result.fetchone())
except Exception as e:
    print("Connection failed:", e)

MySQL Connection successful!
Result: (1,)


In [43]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT DATABASE();"))
    print(result.fetchone())

('ai_jobs',)


In [44]:
df = pd.read_sql("SELECT * FROM ai_jobs_feature_engineered LIMIT 10;",engine)
df.head()

,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,...,salary_range_usd,is_fully_remote,posting_quarter,requires_python,requires_sql,requires_machine_learning,requires_deep_learning,requires_generative_ai,salary_per_experience,is_remote_work
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7.0,Master's,239000.0,155000,290000,Boston,...,135000,False,Q<NA>,True,True,False,False,False,34142.857143,False
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2.0,Bachelor's,166000.0,90000,200000,London,...,110000,False,Q<NA>,True,False,False,False,False,83000.000000,True
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4.0,Associate's,360000.0,160000,300000,Seattle,...,140000,False,Q<NA>,True,False,False,False,False,90000.000000,False
3,AIJOB0004,Data Engineer (AI),Data Engineering,Senior (6-9 yrs),3.0,Bachelor's,161000.0,130000,220000,Singapore,...,90000,False,Q<NA>,True,True,False,False,False,53666.666667,False
4,AIJOB0005,AI Product Manager,Product,Lead (10+ yrs),5.0,Bootcamp/Self-taught,283000.0,140000,260000,Los Angeles,...,120000,False,Q<NA>,False,False,False,False,False,56600.000000,False


In [45]:
query = """
SELECT *
FROM ai_jobs_feature_engineered
LIMIT 10;
"""

df = pd.read_sql(query, engine)

df.head()

,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,...,salary_range_usd,is_fully_remote,posting_quarter,requires_python,requires_sql,requires_machine_learning,requires_deep_learning,requires_generative_ai,salary_per_experience,is_remote_work
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7.0,Master's,239000.0,155000,290000,Boston,...,135000,False,Q<NA>,True,True,False,False,False,34142.857143,False
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2.0,Bachelor's,166000.0,90000,200000,London,...,110000,False,Q<NA>,True,False,False,False,False,83000.000000,True
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4.0,Associate's,360000.0,160000,300000,Seattle,...,140000,False,Q<NA>,True,False,False,False,False,90000.000000,False
3,AIJOB0004,Data Engineer (AI),Data Engineering,Senior (6-9 yrs),3.0,Bachelor's,161000.0,130000,220000,Singapore,...,90000,False,Q<NA>,True,True,False,False,False,53666.666667,False
4,AIJOB0005,AI Product Manager,Product,Lead (10+ yrs),5.0,Bootcamp/Self-taught,283000.0,140000,260000,Los Angeles,...,120000,False,Q<NA>,False,False,False,False,False,56600.000000,False


In [46]:
df.columns.tolist()

['job_id',
 'job_title',
 'job_category',
 'experience_level',
 'years_of_experience',
 'education_required',
 'annual_salary_usd',
 'salary_min_usd',
 'salary_max_usd',
 'city',
 'country',
 'remote_work',
 'company_size',
 'industry',
 'required_skills',
 'ai_salary_premium_pct',
 'demand_score',
 'demand_growth_yoy_pct',
 'benefits_score_10',
 'posting_year',
 'posting_month',
 'is_senior',
 'is_remote_friendly',
 'is_llm_role',
 'salary_tier',
 'experience_group',
 'salary_band',
 'salary_range_usd',
 'is_fully_remote',
 'posting_quarter',
 'requires_python',
 'requires_sql',
 'requires_machine_learning',
 'requires_deep_learning',
 'requires_generative_ai',
 'salary_per_experience',
 'is_remote_work']

In [47]:
# Q1. Total number of job postings
query = """
SELECT COUNT(*) AS total_job_postings
FROM ai_jobs_feature_engineered
"""
result = pd.read_sql(query, engine)

result

,total_job_postings
0,1500


In [48]:
# Q2. Average annual salary
query = """
SELECT 
    ROUND(AVG(annual_salary_usd), 2) AS avg_salary
FROM ai_jobs_feature_engineered;
"""

result = pd.read_sql(query, engine)
result

,avg_salary
0,194892.0


In [49]:
# Q3. Minimum and maximum salary
query = """
SELECT 
    MIN(annual_salary_usd) AS minimum_salary,
    MAX(annual_salary_usd) AS maximum_salary
FROM ai_jobs_feature_engineered;
"""

result = pd.read_sql(query, engine)
result

,minimum_salary,maximum_salary
0,90000.0,384000.0


In [50]:
# Q4. Average years of experience required
query = """
SELECT 
    ROUND(AVG(years_of_experience), 2) AS avg_experience
FROM ai_jobs_feature_engineered;
"""

result = pd.read_sql(query, engine)
result

,avg_experience
0,6.22


In [51]:
# Q5. Number of jobs by category
query = """
SELECT
    job_category,
    COUNT(*) AS total_jobs
FROM ai_jobs_feature_engineered
GROUP BY job_category
ORDER BY total_jobs DESC;
"""
result = pd.read_sql(query, engine)
result

,job_category,total_jobs
0,AI Engineering,736
1,Data Science,127
2,Governance,122
3,Robotics,74
4,Product,70
5,Business,62
6,Infrastructure,55
7,Architecture,52
8,Data Engineering,51
9,ML Operations,51


In [52]:
# Q6. Top 10 jobs titles
query = """
SELECT
    job_title,
    COUNT(*) AS total_jobs
FROM ai_jobs_feature_engineered
GROUP BY job_title
ORDER BY total_jobs DESC
LIMIT 10;
"""
result = pd.read_sql(query, engine)
result

,job_title,total_jobs
0,LLM Engineer,75
1,Robotics Engineer (AI),74
2,Prompt Engineer,71
3,Generative AI Engineer,71
4,AI Product Manager,70
5,Multimodal AI Engineer,67
6,Senior Data Scientist,66
7,AI Compliance Manager,66
8,Senior ML Engineer,64
9,AI Engineer,64


In [53]:
# Q7. Jobs by experience level
query = """
SELECT 
    experience_level,
    COUNT(*) AS total_jobs
FROM ai_jobs_feature_engineered
GROUP BY experience_level
ORDER BY total_jobs DESC;
"""

result = pd.read_sql(query, engine)
result

,experience_level,total_jobs
0,Entry (0-2 yrs),385
1,Lead (10+ yrs),381
2,Mid (3-5 yrs),370
3,Senior (6-9 yrs),364


In [54]:
# Q8. Top 10 countries by job postings
query = """
SELECT 
    country,
    COUNT(*) AS total_jobs
FROM ai_jobs_feature_engineered
GROUP BY country
ORDER BY total_jobs DESC
LIMIT 10;
"""
result = pd.read_sql(query, engine)
result

,country,total_jobs
0,Usa,515
1,Uk,90
2,China,87
3,Canada,85
4,Global,82
5,Germany,81
6,Australia,78
7,Japan,76
8,Switzerland,76
9,Netherlands,74


In [55]:
# Q9. Top 10 cities by job postings

query = """
SELECT
    city,
    COUNT(*) AS total_jobs
FROM ai_jobs_feature_engineered
GROUP BY city
ORDER BY total_jobs DESC
LIMIT 10;
"""

result = pd.read_sql(query, engine)
result

,city,total_jobs
0,San Francisco,92
1,London,90
2,Beijing,87
3,Toronto,85
4,Remote,82
5,Berlin,81
6,Sydney,78
7,Chicago,77
8,Austin,77
9,Tokyo,76


In [56]:
# Q10. Average salary by country

query = """
SELECT 
    country,
    COUNT(*) AS total_jobs,
    ROUND(AVG(annual_salary_usd), 2) AS avg_salary
FROM ai_jobs_feature_engineered
GROUP  BY country
HAVING COUNT(*) >= 5
ORDER BY avg_salary DESC;
"""

result = pd.read_sql(query, engine)
result

,country,total_jobs,avg_salary
0,Usa,515,226189.90
1,Global,82,222043.90
2,Uae,62,194225.81
3,Switzerland,76,190592.11
4,Australia,78,188000.00
5,France,66,183151.52
6,Singapore,71,181830.99
7,Germany,81,181180.25
8,Uk,90,180644.44
9,Canada,85,180588.24


In [57]:
# Q11. Average salary by job category

query = """
SELECT
    job_category,
    COUNT(*) AS total_jobs,
    ROUND(AVG(annual_salary_usd), 2) AS average_salary
FROM ai_jobs_feature_engineered
GROUP BY job_category
ORDER BY average_salary DESC;
"""

result = pd.read_sql(query, engine)
result

,job_category,total_jobs,average_salary
0,Architecture,52,251576.92
1,AI Engineering,736,207982.34
2,Infrastructure,55,203527.27
3,Security,50,200400.00
4,ML Operations,51,199215.69
5,Product,70,194571.43
6,Research,50,192280.00
7,Data Science,127,181275.59
8,Data Engineering,51,176156.86
9,Robotics,74,170851.35


In [58]:
# Q12. Average salary by experience level
query = """
SELECT
    experience_level,
    COUNT(*) AS total_jobs,
    ROUND(AVG(annual_salary_usd), 2) AS average_salary
FROM ai_jobs_feature_engineered
GROUP BY experience_level
ORDER BY average_salary DESC;
"""

result = pd.read_sql(query, engine)
result

,experience_level,total_jobs,average_salary
0,Lead (10+ yrs),381,240055.12
1,Senior (6-9 yrs),364,214280.22
2,Mid (3-5 yrs),370,175983.78
3,Entry (0-2 yrs),385,150038.96


In [59]:
# Q13. Salary band distribution
query = """
SELECT
    salary_band,
    COUNT(*) AS total_jobs
FROM ai_jobs_feature_engineered
GROUP BY salary_band
ORDER BY total_jobs DESC;
"""

result = pd.read_sql(query, engine)
result

,salary_band,total_jobs
0,Very High,1062
1,High,400
2,Medium,38


In [60]:
# Q14. Python requirement
query = """
SELECT
    requires_python,
    COUNT(*) AS total_jobs
FROM ai_jobs_feature_engineered
GROUP BY requires_python
ORDER BY total_jobs DESC;
"""

result = pd.read_sql(query, engine)
result

,requires_python,total_jobs
0,True,942
1,False,558


In [61]:
# Q15. SQL requirement
query = """
SELECT
    requires_sql,
    COUNT(*) AS total_jobs
FROM ai_jobs_feature_engineered
GROUP BY requires_sql
ORDER BY total_jobs DESC;
"""

result = pd.read_sql(query, engine)
result

,requires_sql,total_jobs
0,False,1078
1,True,422


In [62]:
# Q16. Machine Learning requirement
query = """
SELECT
    requires_machine_learning,
    COUNT(*) AS total_jobs
FROM ai_jobs_feature_engineered
GROUP BY requires_machine_learning
ORDER BY total_jobs DESC;
"""

result = pd.read_sql(query, engine)
result

,requires_machine_learning,total_jobs
0,False,1500


In [63]:
# Q17. Deep Learning requirement
query = """
SELECT
    requires_deep_learning,
    COUNT(*) AS total_jobs
FROM ai_jobs_feature_engineered
GROUP BY requires_deep_learning
ORDER BY total_jobs DESC;
"""

result = pd.read_sql(query, engine)
result

,requires_deep_learning,total_jobs
0,False,1409
1,True,91


In [64]:
# Q18. Generative AI requirement
query = """
SELECT
    requires_generative_ai,
    COUNT(*) AS total_jobs
FROM ai_jobs_feature_engineered
GROUP BY requires_generative_ai
ORDER BY total_jobs DESC;
"""

result = pd.read_sql(query, engine)
result


,requires_generative_ai,total_jobs
0,False,1442
1,True,58


In [65]:
#  Q19. Remote vs non-remote jobs
query = """
SELECT
    remote_work,
    COUNT(*) AS total_jobs
FROM ai_jobs_feature_engineered
GROUP BY remote_work
ORDER BY total_jobs DESC;
"""

result = pd.read_sql(query, engine)
result

,remote_work,total_jobs
0,Hybrid,686
1,Fully Remote,445
2,On-Site,369


In [66]:
# Q20. Average salary: remote vs non-remote
query = """
SELECT
    remote_work,
    COUNT(*) AS total_jobs,
    ROUND(AVG(annual_salary_usd), 2) AS average_salary
FROM ai_jobs_feature_engineered
GROUP BY remote_work
ORDER BY average_salary DESC;
"""

result = pd.read_sql(query, engine)
result

,remote_work,total_jobs,average_salary
0,Fully Remote,445,197967.19
1,On-Site,369,195268.83
2,Hybrid,686,192694.46


In [67]:
# Q21. LLM vs non-LLM roles
query = """
SELECT
    is_llm_role,
    COUNT(*) AS total_jobs
FROM ai_jobs_feature_engineered
GROUP BY is_llm_role
ORDER BY total_jobs DESC;
"""

result = pd.read_sql(query, engine)
result

,is_llm_role,total_jobs
0,0,1173
1,1,327


In [68]:
# Q22. LLM vs non-LLM salary comparison
query = """
SELECT
    is_llm_role,
    COUNT(*) AS total_jobs,
    ROUND(AVG(annual_salary_usd), 2) AS average_salary
FROM ai_jobs_feature_engineered
GROUP BY is_llm_role
ORDER BY average_salary DESC;
"""

result = pd.read_sql(query, engine)
result

,is_llm_role,total_jobs,average_salary
0,1,327,207746.18
1,0,1173,191308.61


In [69]:
# Q22. Job categories with high demand and high salary
query = """
SELECT
    job_category,
    COUNT(*) AS total_jobs,
    ROUND(AVG(annual_salary_usd), 2) AS average_salary
FROM ai_jobs_feature_engineered
GROUP BY job_category
HAVING COUNT(*) >= 10
ORDER BY total_jobs DESC, average_salary DESC;
"""

result = pd.read_sql(query, engine)
result

,job_category,total_jobs,average_salary
0,AI Engineering,736,207982.34
1,Data Science,127,181275.59
2,Governance,122,152516.39
3,Robotics,74,170851.35
4,Product,70,194571.43
5,Business,62,134145.16
6,Infrastructure,55,203527.27
7,Architecture,52,251576.92
8,ML Operations,51,199215.69
9,Data Engineering,51,176156.86
